# CluedIn instance status report

A summary of the golden records on an instance. Set `ENTITY_TYPE_PREFIX`
to limit the report to your own entity types, for example `/Example`.


In [ ]:
# The notebooks run from the repository root so that the modules import.
import os
import sys
import warnings

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# Every input comes from the metrics configuration -- no hard-coded entity
# types or vocabulary keys. Point METRICS_CONFIG at your own file.
CONFIG_PATH = os.getenv('METRICS_CONFIG', 'config/metrics.example.json')
CONFIG_PATH


In [ ]:
%pip install matplotlib


In [ ]:
import cluedin

from api import get_context

ENTITY_TYPE_PREFIX = os.getenv('ENTITY_TYPE_PREFIX', '')
context = get_context()


## Golden records by entity type


In [ ]:
SUMMARY_QUERY = '''
query aggregateQuery($query: String, $filters: [FilterQuery]) {
  aggregate(query: $query, filters: $filters) {
    aggregations
    totalResults
  }
}
'''

summary = cluedin.gql.gql(context, SUMMARY_QUERY, {'query': '*'})
aggregations = summary['data']['aggregate']['aggregations']

counts = {
    item['Name']: item['Count']
    for item in aggregations['entityType']['Items']
    if item['Name'].startswith(ENTITY_TYPE_PREFIX)
}

print(f'{sum(counts.values())} golden records in {len(counts)} entity types')


In [ ]:
import matplotlib.pyplot as plt

def short(name):
    return name[len(ENTITY_TYPE_PREFIX):].lstrip('/') or name

ordered = sorted(counts.items(), key=lambda item: -item[1])

plt.figure(figsize=(min(25, max(8, len(ordered))), 5))
plt.bar([short(name) for name, _ in ordered],
        [count for _, count in ordered])
for index, (_, count) in enumerate(ordered):
    plt.text(index, count, count, ha='center', va='bottom')
plt.title('Golden records by entity type')
plt.xlabel('Entity type')
plt.ylabel('Golden records')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Entity code origins


In [ ]:
origins = aggregations.get('origin', {}).get('Items', [])

for item in sorted(origins, key=lambda i: -i['Count']):
    print(f"{item['Count']:>10}  {item['Name']}")
